In [1]:
#1. libraries
import urllib.request
import re
from urllib.request import Request, urlopen, urlretrieve
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import math
import time
import pickle
import zipfile
from zipfile import ZipFile
from pathlib import Path
from datetime import datetime, timedelta
import os
from zoneinfo import ZoneInfo
print('loaded')



loaded


In [2]:
#2. global definitions

skip_yesterday = False #when True rebuild the database. when False run one day and add to the db.  

def is_databricks():
    return "DATABRICKS_RUNTIME_VERSION" in os.environ

in_databricks = is_databricks()

if in_databricks == True:
    base = Path('/Volumes/exploration/bills_repository/files/')
    #%pip install numpy==2.1.3   
else:
    base = Path("C:/Users/BillNixey/OneDrive - Squadron Energy/Desktop/Working_files/New_model/test/")

if skip_yesterday == True:
    days=50 #rebuild the database. Must be 2 or greater
   
else:
    days = 1 #run one day and add to the db


allowed_fuels = ['Wind','Battery','Solar','Black Coal','Other','Gas','Hydro','Natural Gas (Pipeline)','Diesel','Brown Coal']

now = datetime.now(ZoneInfo("Australia/Brisbane"))
s1 = str(now)
s2 = s1[:11]
s2 = s2.replace(":", "")
s2 = s2.replace(" ", "")

#url2 = "https://www.neopoint.com.au/Service/Csv?f=107%20Information%5CPlantInformation&from="+s2+"%2000%3A00&period=Daily&instances=&section=-1&key=squnix77"
#df_duid = pd.read_csv(url2)

try:
    url2 = (
        f"https://www.neopoint.com.au/Service/Csv?"
        f"f=107%20Information%5CPlantInformation"
        f"&from={s2}%2000%3A00"
        f"&period=Daily&instances=&section=-1&key=squnix77"
    )

    df_duid = pd.read_csv(url2)
    df_duid = df_duid.rename(columns={"TransmissionLossFactor": "MLF"}) 
    df_duid = df_duid[['DUID','REGIONID','MLF','FUEL']]
    exceptions = set(df_duid['FUEL'].dropna().unique()) - set(allowed_fuels)

    if exceptions:
        print("Unexpected FUEL values found:")
        for fuel in sorted(exceptions):
            print(f"  {fuel}")
    else:
        print("All FUEL values are valid.")
    
    df_duid['FUEL'] = df_duid['FUEL'].replace('Natural Gas (Pipeline)', 'Gas')
    df_duid["FUEL"] = df_duid["FUEL"].str.replace(" ", "_", regex=False)

    if in_databricks == True:
       df_duid.to_parquet(base /"df_duid_LIVE.parquet", index=False)
    else:
       df_duid.to_csv(base / 'df_duid_LIVE.csv' , index=False)

except Exception as e:
    print(f"Failed to load DUID data: {e}")
   
print('done')


All FUEL values are valid.
done


In [3]:
#3. functions

def latest_file(url,penultimate,type): 
    my_list = []
    req = Request(url)
    a = urlopen(req).read()
    soup = BeautifulSoup(a, 'html.parser')
    x = (soup.find_all('a')) #read all on html page
    #this loop finds the most recent file
    for i in x:    
        file_name = i.extract().get_text()       #take only file names
        if(file_name[-3:]=='zip'):            #get only zip files 
                m = re.search(r"\d", file_name) # this finds first digit in string
                date = int(file_name[m.start():m.start()+8]) #grabs the date and converts to an integer          
                my_list.append(date) #add item to array
    if(type=='max'):
        my_value = max(my_list) #find most recent report in array
    else:
        my_value = int(type) #find a specified day
    location = my_list.index(my_value)    #position in array of latest date. Note 'zero' position at start of list.
    #the loop below downloads latest file a file and splits it into two df. One each for price and volume. Repeats for next most recent file.   
    my_file = x[location+len(x)-len(my_list)+penultimate].extract().get_text() #This "len(x)-len(my_list)" takes into account 2 extra values in the x array
    return my_file

def download_files(d,url1,base):
    files = []
    if skip_yesterday == True:
        my_val = -1
    else:
        my_val = 0
    #print(my_val,-d,-1)
    for n in range(my_val,-d,-1):      
        #specify type as most recent 'max' or as specific file name date 
        my_file = latest_file(url1,n,'max')    
        #download and extract file
        print('Downloading file from http: '+my_file)
        urllib.request.urlretrieve(url1+my_file, base / my_file)        
        with ZipFile(base / my_file, 'r') as z:
            z.extractall(base)
        file_path = Path(base / my_file)
        file_path.unlink()   
        files = files + [my_file[:-3] + "CSV"]
    
    print("folder contains",len(files),"files")
    return files

def find_row2(my_file,label):
    row=0
    with open(my_file) as f:  
        for line in f:
            row_data = line.split(',')
            row= row+1
            if row_data[2] == label:
                return row
        return row
print('loaded')


loaded


In [4]:
#4. Create pickle [SETTLEMENTDATE, DUID, PRICE, VOL, GEN/LOAD] from bid files
#volumes

start = time.time()
print("Starting...")

url1='https://www.nemweb.com.au/REPORTS/CURRENT/Bidmove_Complete/'
files = download_files(days,url1,base)

cols0 = [f'BANDAVAIL{i}' for i in range(1, 11)] 
cols1 = cols0 + ['MAXAVAIL']

base_cols = ['LASTCHANGED','INTERVAL_DATETIME','SETTLEMENTDATE','DUID','BIDTYPE','DIRECTION']
# include the first column used for filtering (table name column)
usecols = ['BIDPEROFFER_D'] + base_cols + cols1
frames = []


for b in files:
    #print(b)     
    counter= find_row2(base / b,"BIDPEROFFER_D")
    df=pd.read_csv(base / b,skiprows=counter-1,dtype=object)
    
    df = df[(df['BIDPEROFFER_D'] == 'BIDPEROFFER_D') &  (df['BIDTYPE'] == 'ENERGY') ]
    df['SETTLEMENTDATE'] = pd.to_datetime(df['SETTLEMENTDATE'])
    df['LASTCHANGED'] = pd.to_datetime(df['LASTCHANGED'])
    df['INTERVAL_DATETIME'] = pd.to_datetime(df['INTERVAL_DATETIME'])
    df = df[df['INTERVAL_DATETIME'].dt.minute.isin([0, 30])] ###################### filter out non half-hours
    

    for c in cols1:
        df[c] = pd.to_numeric(df[c], errors='coerce')
        # Keep latest LASTCHANGED per DUID+INTERVAL_DATETIME
    
    df = df.sort_values(['DUID','INTERVAL_DATETIME','LASTCHANGED'])
    df = df.drop_duplicates(subset=['DUID','INTERVAL_DATETIME','DIRECTION'], keep='last')
    df = df[df['MAXAVAIL'] > 0] 
    
    df = df.loc[:, base_cols + cols1]
    frames.append(df)

df2 = pd.concat(frames, ignore_index=True)
df2[cols1] = df2[cols1].fillna(0.0) #tidy
duids = list(df_duid[(df_duid.FUEL == "Wind") | (df_duid.FUEL == "Solar")].DUID)
df2 = df2[~df2.DUID.isin(duids)].reset_index(drop=True) #remove wind/solar#########################

# trim bands to MAXAVAIL
df2["MAXAVAIL2"] = df2["MAXAVAIL"]

bands = df2[cols0].to_numpy(dtype=float)              # shape (n,10)
cap = df2["MAXAVAIL"].to_numpy(dtype=float)                  # shape (n,)

# If MAXAVAIL is missing, treat as "no cap" (leave row unchanged)
row_total = bands.sum(axis=1)
cap_eff = np.where(np.isfinite(cap), np.maximum(cap, 0.0), row_total)

# Allocate MW from low → high bands until cap is filled (equivalent to cutting high bands first)
remaining = cap_eff.copy()
new_bands = np.zeros_like(bands)

for j in range(bands.shape[1]):  # BANDAVAIL1 .. BANDAVAIL10
    take = np.minimum(bands[:, j], remaining)
    new_bands[:, j] = take
    remaining -= take
    remaining = np.maximum(remaining, 0.0)

df2[cols0] = new_bands

# Optional checks
df2["BANDAVAIL_TOTAL_BEFORE"] = row_total
df2["BANDAVAIL_TOTAL_AFTER"]  = df2[cols0].sum(axis=1)



df_bid_volumes = df2.copy()

print('Time to complete:', time.time()-start, 'seconds.')
#df2.head()

Starting...
folder contains 1 files
Time to complete: 5.0125041007995605 seconds.


In [5]:
#df_bid_volumes[(df_bid_volumes.DUID == 'ORABESS1') & (df_bid_volumes.INTERVAL_DATETIME == '	2026-06-15 17:00:00')]

In [6]:
#5. get energy bid prices from DUIDs

start = time.time()
print("Starting...")

df_bid_prices_parts = []

for b in files:
    print(b)     
    counter= find_row2(base / b,"BIDPEROFFER_D")
    df=pd.read_csv(base / b,skiprows=1,nrows=counter-3,dtype=object)
    file_path = Path(base / b)
    file_path.unlink()
    
    df = df.loc[(df['BIDDAYOFFER_D'] == 'BIDDAYOFFER_D') & (df['BIDTYPE']=='ENERGY') ]
    df = df.loc[:,('LASTCHANGED','SETTLEMENTDATE','DUID','BIDTYPE','DIRECTION','PRICEBAND1','PRICEBAND2','PRICEBAND3','PRICEBAND4','PRICEBAND5','PRICEBAND6','PRICEBAND7','PRICEBAND8','PRICEBAND9','PRICEBAND10')] 
    df = df.assign(LASTCHANGED = pd.to_datetime(df['LASTCHANGED']))
    df = df.assign(SETTLEMENTDATE = pd.to_datetime(df['SETTLEMENTDATE']))
    colsA = ['PRICEBAND1','PRICEBAND2','PRICEBAND3','PRICEBAND4','PRICEBAND5','PRICEBAND6','PRICEBAND7','PRICEBAND8','PRICEBAND9','PRICEBAND10']
    df[colsA] = df[colsA].apply(pd.to_numeric)
       
    #find most recent price and volume offers
    df_test = (df.sort_values('LASTCHANGED', ascending=False).drop_duplicates(subset=['DUID','DIRECTION'], keep='first').reset_index(drop=True)) #
    df_bid_prices_parts.append(df_test)

df_bid_prices = pd.concat(df_bid_prices_parts, ignore_index=True)
        

print('Time to complete:', time.time()-start, 'seconds.')



Starting...
PUBLIC_BIDMOVE_COMPLETE_20260811_0000000532197880.CSV
Time to complete: 0.07692503929138184 seconds.


In [7]:
#6. Finish: Create parquet [SETTLEMENTDATE, DUID, PRICE, VOL, GEN/LOAD] from bid files



start = time.time()
print("Starting...")

df2 = df_bid_volumes.copy()
df3 = df_bid_prices.copy()

#merge price and volume
df4 = pd.merge(df2,df3,on=['SETTLEMENTDATE','DUID','BIDTYPE','DIRECTION'],how='left')
df4.head()

df = df4.copy()

# 1) column lists
price_cols = [f"PRICEBAND{i}" for i in range(1, 11)]
vol_cols   = [f"BANDAVAIL{i}" for i in range(1, 11)]

# 2) ensure datetime + numeric
df["INTERVAL_DATETIME"] = pd.to_datetime(df["INTERVAL_DATETIME"])
df[price_cols] = df[price_cols].apply(pd.to_numeric)
df[vol_cols]   = df[vol_cols].apply(pd.to_numeric).fillna(0.0)

# 3) melt price and volume, extract band number, then merge
p = df.melt(
    id_vars=["INTERVAL_DATETIME", "DUID", "DIRECTION"],
    value_vars=price_cols,
    var_name="BAND",
    value_name="PRICE"
)
p["BAND"] = p["BAND"].str.extract(r"(\d+)$").astype(int)

v = df.melt(
    id_vars=["INTERVAL_DATETIME", "DUID", "DIRECTION"],
    value_vars=vol_cols,
    var_name="BAND",
    value_name="VOL"
)
v["BAND"] = v["BAND"].str.extract(r"(\d+)$").astype(int)

out = p.merge(v, on=["INTERVAL_DATETIME", "DUID", "DIRECTION", "BAND"], how="inner")

# 4) GEN / LOAD flag (based on DIRECTION)
out["GEN_LOAD"] = out["DIRECTION"].map({"GEN": "GEN", "LOAD": "LOAD"}).fillna(out["DIRECTION"])

# drop zero volume bands and/or missing prices
out = out[(out["VOL"] > 0) & (out["PRICE"].notna())].reset_index(drop=True)

#add regions and apply MLFs

out = pd.merge(out,df_duid, on='DUID',how = 'left')
out['PRICE'] = out['PRICE'] / out['MLF']
out = out[["INTERVAL_DATETIME", "DUID", "PRICE", "VOL", "GEN_LOAD",'REGIONID','FUEL']]
#out.to_csv(base /"Bid_data_LIVE.csv", index=False)


if skip_yesterday == True:
    final_df = out
else:
    
    existing_df = pd.read_parquet(base / "Bid_data_LIVE.parquet")
    final_df = pd.concat([existing_df, out], ignore_index=True)
     # get earliest day
    earliest_day = final_df['INTERVAL_DATETIME'].dt.floor('D').min()
    # drop rows from that day
    final_df = final_df[final_df['INTERVAL_DATETIME'].dt.floor('D') != earliest_day]
    
#final save
final_df.to_parquet(base /"Bid_data_LIVE.parquet", index=False)

print("max date:", final_df.INTERVAL_DATETIME.max(),"min date:",final_df.INTERVAL_DATETIME.min())
print('Time to complete:', time.time()-start, 'seconds.')

final_df.head()



Starting...
max date: 2026-08-12 04:00:00 min date: 2026-06-12 00:00:00
Time to complete: 1.7758679389953613 seconds.


,INTERVAL_DATETIME,DUID,PRICE,VOL,GEN_LOAD,REGIONID,FUEL
0,2026-06-14 04:30:00,BALB1,-995.616271,30.0,LOAD,VIC1,Battery
1,2026-06-14 05:00:00,BALB1,-995.616271,30.0,LOAD,VIC1,Battery
2,2026-06-14 05:30:00,BALB1,-995.616271,30.0,LOAD,VIC1,Battery
3,2026-06-14 06:00:00,BALB1,-995.616271,30.0,LOAD,VIC1,Battery
4,2026-06-14 06:30:00,BALB1,-995.616271,30.0,LOAD,VIC1,Battery


In [8]:
#7. get demand data from public_prices


start = time.time()
print("Starting...")

url1='https://www.nemweb.com.au/REPORTS/CURRENT/Public_Prices/'
files = download_files(days,url1,base)

df2_parts = [] 
for b in files:
    print(b)
    counter = 1440      
    df=pd.read_csv(base / b,skiprows=1,nrows=counter)
    file_path = Path(base / b)
    file_path.unlink()
    df = df.loc[df['DREGION'] == 'DREGION']
    df = df.assign(SETTLEMENTDATE = pd.to_datetime(df['SETTLEMENTDATE']))   
    df = df.sort_values(['SETTLEMENTDATE','REGIONID']) 
    df = df.loc[:,('SETTLEMENTDATE','REGIONID','TOTALDEMAND','DISPATCHABLELOAD','AVAILABLEGENERATION','AVAILABLELOAD','DISPATCHABLEGENERATION','NETINTERCHANGE')]
    df= df.assign(SURPLUSCAPACITY = df['AVAILABLEGENERATION'] - df['TOTALDEMAND']  )#- df['DISPATCHABLELOAD']
    df = df[df['SETTLEMENTDATE'].dt.minute.isin([0, 30])] ###################### filter out non half-hours
    df2_parts.append(df)
df2 = pd.concat(df2_parts, ignore_index=True)


df2["SETTLEMENTDATE"] = pd.to_datetime(df2["SETTLEMENTDATE"])
df2 = df2.sort_values(by=["SETTLEMENTDATE", "REGIONID"]).reset_index(drop=True)


if skip_yesterday == True:
    combined_df = df2   
else:
    
    existing_df = pd.read_parquet(base / "spare_capacity_LIVE.parquet")
    combined_df = pd.concat([existing_df, df2], ignore_index=True)
    # get earliest day
    earliest_day = combined_df['SETTLEMENTDATE'].dt.floor('D').min()
    # drop rows from that day
    combined_df = combined_df[combined_df['SETTLEMENTDATE'].dt.floor('D') != earliest_day]
    

combined_df.to_parquet(base /"spare_capacity_LIVE.parquet", index=False)

print("max date:", combined_df.SETTLEMENTDATE.max(),"min date:",combined_df.SETTLEMENTDATE.min())

print('Time to complete:', time.time()-start, 'seconds.')


Starting...
folder contains 1 files
PUBLIC_PRICES_202608110000_20260812040507.CSV
max date: 2026-08-12 04:00:00 min date: 2026-06-12 00:00:00
Time to complete: 2.2869791984558105 seconds.


In [9]:
#8. get actual wind solar total cleared from https://www.nemweb.com.au/Reports/CURRENT/Next_Day_Dispatch



start = time.time()
print("Starting...")

url1='https://www.nemweb.com.au/Reports/CURRENT/Next_Day_Dispatch/'
files = download_files(days,url1,base)

df2_parts = []
for b in files:
    print(b)
    counter = find_row2(base / b,'LOCAL_PRICE')
    df=pd.read_csv(base / b,skiprows=1,nrows=counter-1,dtype=object)
    file_path = Path(base / b)
    file_path.unlink()
    df = df.loc[df['UNIT_SOLUTION'] == 'UNIT_SOLUTION']
    df = df.assign(SETTLEMENTDATE = pd.to_datetime(df['SETTLEMENTDATE']))   
    df = df[df['SETTLEMENTDATE'].dt.minute.isin([0, 30])] #### filter out non half-hours
    df2_parts.append(df)
df2 = pd.concat(df2_parts, ignore_index=True)

df2 = pd.merge(df2,df_duid, on='DUID',how = 'left')
df2 = df2[df2.FUEL.isin(['Wind','Solar'])]
#df2 = df2[df2.UIGF >0] not needed, but alternative method
df2 = df2.loc[:,('SETTLEMENTDATE','DUID','FUEL','REGIONID','UIGF')]
df2['UIGF'] = df2['UIGF'].apply(pd.to_numeric)#,errors='coerce'
df2 =  (df2.groupby(['SETTLEMENTDATE', 'FUEL', 'REGIONID'],as_index=False)['UIGF'].sum())

df2 = df2.pivot(index=['SETTLEMENTDATE', 'REGIONID'],columns='FUEL',values='UIGF').fillna(0).reset_index()

df2 = df2.sort_values(by=["SETTLEMENTDATE","REGIONID"]).reset_index(drop=True)


if skip_yesterday == True:
    combined_df = df2   
else:
    
    existing_df = pd.read_parquet(base / "Actual_wind_solar_LIVE.parquet")
    combined_df = pd.concat([existing_df, df2], ignore_index=True)
    # get earliest day
    earliest_day = combined_df['SETTLEMENTDATE'].dt.floor('D').min()
    # drop rows from that day
    combined_df = combined_df[combined_df['SETTLEMENTDATE'].dt.floor('D') != earliest_day]
    

combined_df.to_parquet(base /"Actual_wind_solar_LIVE.parquet", index=False)

print("max date:", combined_df.SETTLEMENTDATE.max(),"min date:",combined_df.SETTLEMENTDATE.min())

print('Time to complete:', time.time()-start, 'seconds.')


Starting...
folder contains 1 files
PUBLIC_NEXT_DAY_DISPATCH_20260811_0000000532194219.CSV
max date: 2026-08-12 04:00:00 min date: 2026-06-08 00:00:00
Time to complete: 4.743617296218872 seconds.
